# Transfer Learning

**Instructor:** Amna Mazen

# Transfer learning
A machine learning technique where a model trained on one task is used as a starting point for a related task. In the context of CNNs, this involves leveraging the knowledge learned from a large dataset to solve a new problem with limited data.

![Image](https://raw.githubusercontent.com/MazenMTULab/ML_COURSE_RESOURCES/refs/heads/main/Figs/Transfer%20Learning.webp)


**Why Transfer Learning for CNNs?**

- **Limited Data:** CNNs require large datasets for optimal performance. Transfer learning allows us to train effective models with smaller datasets.

- **Computational Efficiency:** Pre-trained models can be fine-tuned more efficiently than training a model from scratch.

- **Improved Accuracy:** Transfer learning often leads to better performance, especially for tasks with similar characteristics to the original training data.




**Common Transfer Learning Approaches for CNNs**

**Feature Extraction:**

**Frozen Layers:** The initial layers of a pre-trained CNN are frozen, and only the final layers are trained on the new task. This assumes that the *early layers extract generic features* that are useful for a variety of tasks.

**Fine-tuning:** Some layers are fine-tuned along with the new layers. This allows the network to adapt to the specific characteristics of the new task.



![Image](https://raw.githubusercontent.com/MazenMTULab/ML_COURSE_RESOURCES/refs/heads/main/Figs/Transfer%20learning%202.webp)

Image Source: https://medium.com/@saba99/transfer-learning-bbf6b67deb88

**Popular Pre-trained CNN Architectures**
- **VGGNet:** Known for its depth and simplicity.

- **ResNet:** Uses residual connections to overcome the vanishing gradient problem.

- **InceptionNet:** Combines different-sized convolutions to extract features at multiple scales.

- **MobileNet:** Designed for efficient inference on mobile devices.

- **EfficientNet:** A compound scaling method that balances depth, width, and resolution.


**Best Practices for Transfer Learning**
- **Choose a suitable pre-trained model:**Consider the similarity between the original task and your new task.

- **Experiment with different layers to freeze or fine-tune:** Start with freezing more layers and gradually unfreeze them as needed.

- **Adjust the learning rate:** Use a lower learning rate for fine-tuning to avoid overfitting.

- **Data augmentation:** Augment your training data to increase its diversity and prevent overfitting.

- **Regularization techniques:** Employ techniques like dropout or L1/L2 regularization to prevent overfitting.

L1 and L2 regularization are two common techniques used in machine learning and deep learning to prevent overfitting by adding a penalty term to the loss function. They both aim to shrink the model parameters.

**L2 regularization**

- L2 regularization adds a penalty proportional to the square of the magnitude of the model's weights (parameters).

- It penalizes large weights more heavily by adding the sum of their squared values to the loss function.

- This encourages the model to keep the weights small and reduces the complexity of the model, preventing overfitting.

**L1 regularization**

- L1 regularization adds a penalty proportional to the absolute value of the weights.

- It encourages sparsity in the weights, meaning some weights may be exactly zero. This is useful for feature selection as it can automatically eliminate irrelevant features.

## Use Kaggle datasets

If you want to use the **Kaggle** dataset directly in Google Colab without manually downloading it, follow these steps to authenticate with Kaggle and load your dataset programmatically.

### Step 1: Get Kaggle API Token
- Go to https://www.kaggle.com
- Click on your profile icon in the top-right corner and select **settings**
- Scroll down to the **API** section and click "**Create New API Token**"
- A file named **kaggle.json** will be downloaded — do not open it, just upload it in the next step


### Step2: Upload kaggle.json in Colab






In [ ]:
from google.colab import files
files.upload()

TypeError: 'NoneType' object is not subscriptable

After running the cell, click the **"Choose Files"** button in the output area.

Select your kaggle.json file from your computer (downloaded from your Kaggle account).

### Step 3: Set Up Kaggle API Access

In [ ]:
import os
import zipfile

# Create Kaggle directory
os.makedirs("/root/.kaggle", exist_ok=True)

# Move kaggle.json to the right location
!mv kaggle.json /root/.kaggle/

# Set proper permissions
!chmod 600 /root/.kaggle/kaggle.json

### Step 4: Download the Dataset from Kaggle

In [ ]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset

### Step 5: Unzip and Use the Dataset

In [ ]:
# Unzip the dataset
with zipfile.ZipFile("brain-tumor-mri-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/brain-tumor-classification-mri")

### Step 6: Use flow_from_directory with Extracted Folder

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For computing class weights to handle imbalanced datasets
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow and Keras libraries for building models
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras import regularizers

from tensorflow.keras.preprocessing.image import ImageDataGenerator

image_height = 299
image_width = 299
BATCH_SIZE = 48

train_dir = "/content/brain-tumor-classification-mri/Training"

data_generator_1 = ImageDataGenerator(
    rescale=1./255,
    rotation_range=5,
    width_shift_range=0.05,
    height_shift_range=0.05,
    shear_range=0.05,
    zoom_range=0.05,
    brightness_range=[0.95, 1.05],
    fill_mode='nearest'
)

train_generator1 = data_generator_1.flow_from_directory(
    directory=train_dir,
    color_mode="rgb",
    target_size=(image_height, image_width),
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

# Create first data generator with slight augmentations (rotation, shift, etc.)
data_generator_1 = ImageDataGenerator(
    rescale=1./255,              # Normalize pixel values
    rotation_range=5,            # Rotate images by up to 5 degrees
    width_shift_range=0.05,      # Shift images horizontally by up to 5%
    height_shift_range=0.05,     # Shift images vertically by up to 5%
    shear_range=0.05,            # Apply shearing transformations
    zoom_range=0.05,             # Zoom in/out by up to 5%
    brightness_range=[0.95, 1.05],# Slight brightness variation
    horizontal_flip=False,       # Do not flip images horizontally
    vertical_flip=False,         # Do not flip images vertically
    fill_mode='nearest'          # Fill in missing pixels after transformations
)
print('Data Augmentation 1 created')

# Create a second data generator with slightly stronger augmentations (if needed)
data_generator_2 = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1],
    horizontal_flip=False,
    vertical_flip=False,
    fill_mode='nearest'
)
print('Data Augmentation 2 created')

# Create a third data generator for validation/test data without augmentation (only rescaling)
data_generator_3 = ImageDataGenerator(rescale=1./255)



In [ ]:
# Set parameters for image size and batch processing
BATCH_SIZE = 48
image_height = 299
image_width = 299

# Load the training dataset using the first data generator
train_generator_1 = data_generator_1.flow_from_directory(
    directory=train_dir,  # Replace with your training data path
    color_mode="rgb",
    target_size=(image_height, image_width),
    class_mode="categorical",            # Use categorical labels for multi-class classification
    batch_size=BATCH_SIZE,
    shuffle=True,                        # Shuffle data for better training
    seed=42                              # Set seed for reproducibility
)
print('Train data loaded')
print('Data Augmentation 1 was used to generate train data set\n')



In [ ]:
# Load the testing dataset using the third generator (no augmentation)

test_dir = "/content/brain-tumor-classification-mri/Testing"
test_generator = data_generator_3.flow_from_directory(
    directory=test_dir,   # Replace with your testing data path
    color_mode="rgb",
    target_size=(image_height, image_width),
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)
print('Test data loaded')



In [ ]:
# Retrieve the mapping of class indices and print class labels
dict_class = train_generator_1.class_indices
print('Dictionary: {}'.format(dict_class))
class_names = list(dict_class.keys())
print('Class labels: {}'.format(class_names))



**These are the four categories of brain conditions:**

- Glioma (0): A type of tumor that occurs in the brain and spinal cord.

- Meningioma (1): A tumor that arises from the meninges, the membranes surrounding the brain and spinal cord.

- No Tumor (2): Indicates that the brain scan does not show any tumor.

- Pituitary (3): Refers to tumors that develop in the pituitary gland.

In [ ]:
frequency = np.unique(train_generator1.classes, return_counts=True)
plt.title("Trainning dataset", fontsize='20')
plt.pie(frequency[1], labels = class_names, autopct='%1.0f%%');

In [ ]:
# Dataset characteristics
print("Dataset Characteristics of Train Data Set:\n")
print("Number of images:", len(train_generator1.classes))
print("Number of glioma_tumor images:", len([label for label in train_generator1.classes if label == 0]))
print("Number of meningioma_tumor images:", len([label for label in
train_generator1.classes if label == 1]))
print("Number of no_tumor images:", len([label for label in train_generator1.classes if label == 2]))
print("Number of pituitary_tumor images:", len([label for label in train_generator1.classes if label == 3]))
print()
# Dataset characteristics
print("Dataset Characteristics of Test Data Set:\n")
print("Number of images:", len(test_generator.classes))
print("Number of glioma_tumor images:", len([label for label in test_generator.classes if label == 0]))
print("Number of meningioma_tumor images:", len([label for label in test_generator.classes if label == 1]))
print("Number of no_tumor images:", len([label for label in test_generator.classes if label == 2]))
print("Number of pituitary_tumor images:", len([label for label in test_generator.classes if label == 3]))
print()

**Why Compute Class Weights?**

- In an imbalanced dataset, some classes have significantly fewer samples than others. If trained naively, the model may become biased toward the majority class and perform poorly on underrepresented classes.

- To compensate for this imbalance, we assign a higher weight to minority classes so that the model pays more attention to them.

The class weight is calculated as:


$$ w_i = \frac{N}{k \times n_i} $$


where:

$𝑤_
𝑖$
​
  = weight for class
𝑖
,

𝑁
 = total number of samples,

𝑘
 = total number of unique classes,

$𝑛
_𝑖$
​
  = number of samples in class
𝑖
.

Thus, minority classes get higher weights.

Assume class distribution:

Class 0 (glioma): 100 samples

Class 1 (meningioma): 50 samples

Class 2 (notumor): 200 samples

Class 3 (pituitary): 25 samples

The computed class weights might look like:

N=100+50+200+25=375

k=4

w0=375/400=0.9375

w1=375/200=1.875

w2=375/800=0.46875

w3=375/100=3.75


In [ ]:
# Compute class weights to help with imbalanced classes during training
class_weights = compute_class_weight(class_weight="balanced",
                                     classes=np.unique(train_generator_1.classes),
                                     y=train_generator_1.classes)
class_weights = dict(zip(np.unique(train_generator_1.classes), class_weights))
print(class_weights)




In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print('Train image data from Data Augmentation 1')

# Fetch augmented images and labels
img, label = next(train_generator1)

# Set figure size for 3x5 grid
plt.figure(figsize=(15, 9))

# Loop to display 15 images in a 3x5 layout
for i in range(15):
    plt.subplot(3, 5, i+1)
    plt.imshow(img[i])
    plt.axis('off')
    plt.title(class_names[np.argmax(label[i])])

# Show all images at once
plt.show()


## Convolutional neural networks (CNNs)

In [ ]:
# Define the epochs for training
EPOCHS = 2

# Define callbacks for early stopping and learning rate reduction
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
early_stopping = EarlyStopping(monitor='val_accuracy', patience=2, verbose=1, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.001, patience=10, verbose=1)

##**Transfer Learning using VGG16**

VGG16 is a convolutional neural network (CNN) architecture that was introduced in **2014**. It was developed by researchers from the **University of Oxford** and is known for its simplicity and effectiveness in image classification tasks.

VGG16 is named "16" because it has 16 weight layers (i.e., layers with learnable parameters), which include:

- 13 convolutional layers
- 3 fully connected (FC) layers

![Image](https://raw.githubusercontent.com/MazenMTULab/ML_COURSE_RESOURCES/refs/heads/main/Figs/VGG16.webp)

Key features of VGG16:
- **Simple Architecture:** VGG16 uses a stack of 13 convolutional layers, followed by three fully connected layers and a final softmax layer for classification.
- **Small Filters:** The convolutional layers in VGG16 use small 3x3 filters, which are repeated multiple times to increase the depth of the network.

- **Uniform Stride:** The convolutional layers use a uniform stride of 1, which means that the filters are applied to every pixel in the input image.
- **Max Pooling:** After each block of convolutional layers, a max pooling layer is used to reduce the spatial dimensions of the feature maps.

**Imagenet dataset:**

What is the ImageNet Dataset?
The ImageNet dataset is a large-scale visual database designed for image classification, object detection, and computer vision research. It is widely used to train deep learning models, especially convolutional neural networks (CNNs).

**Key Features of ImageNet:**

- Contains 14+ million images

- Over 1 million labeled images for training

- Around 100,000 test images

**Categories (Classes)**

Organized into 1,000 object categories (e.g., dogs, cats, cars, airplanes, etc.).

![Image](https://cs.stanford.edu/people/karpathy/cnnembed/cnn_embed_1k.jpg)

Source: https://cs.stanford.edu/people/karpathy/cnnembed/cnn_embed_1k.jpg

In [ ]:
# Create a MirroredStrategy
import tensorflow as tf
from keras.models import Sequential
from tensorflow.keras.applications import VGG16, ResNet50, InceptionV3, MobileNetV2, DenseNet121
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from keras import regularizers

BATCH_SIZE = 48
image_height = 299
image_width = 299

train_data = train_generator1


# Load the pre-trained VGG16 model without the top classification layer
base_model_VGG16 = VGG16(weights='imagenet', include_top=False, input_shape=(image_height, image_width, 3))
# Set the layers of the base model as non-trainable (freeze them)
for layer in base_model_VGG16.layers:
  layer.trainable = False
# Create a new model and add the VGG16 base model
model_VGG16 = Sequential()
model_VGG16.add(base_model_VGG16)
# Add a fully connected layer and output layer for classification
model_VGG16.add(GlobalAveragePooling2D())
model_VGG16.add(Dense(128, activation='relu',kernel_regularizer=regularizers.l2(0.001)))
model_VGG16.add(Dropout(0.4))
model_VGG16.add(Dense(64, activation='relu',kernel_regularizer=regularizers.l2(0.001)))
model_VGG16.add(Dropout(0.2))
model_VGG16.add(Dense(4, activation='softmax'))
# Model summary
print("Model Summary (VGG16):")
model_VGG16.summary()
print()
# Compile the model
model_VGG16.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Train the model

history_VGG16 = model_VGG16.fit(train_data, epochs=EPOCHS, validation_data=test_generator, callbacks=[early_stopping], class_weight=class_weights)
# Validate the model
val_loss_VGG16, val_accuracy_VGG16 = model_VGG16.evaluate(test_generator, steps=len(test_generator))
print(f'Validation Loss: {val_loss_VGG16:.4f}')
print(f'Validation Accuracy: {val_accuracy_VGG16:.4f}')

##**Transfer Learning using MobileNetV2**

MobileNetV2 is a convolutional neural network (CNN) architecture designed
specifically for mobile and embedded vision applications. It was introduced in **2018** and is known for its high efficiency and accuracy.

In [ ]:
%%time
from tensorflow.keras.applications import MobileNetV2
# Load the pre-trained MobileNetV2 model without the top classification layer
base_model_MobileNet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(image_height, image_width, 3))
# Set the layers of the base model as non-trainable (freeze them)
for layer in base_model_MobileNet.layers:
    layer.trainable = False
# Create a new model and add the MobileNetV2 base model
model_MobileNet = Sequential()
model_MobileNet.add(base_model_MobileNet)
# Add a global average pooling layer and output layer for classification
model_MobileNet.add(GlobalAveragePooling2D())
model_MobileNet.add(Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_MobileNet.add(Dropout(0.4))
model_MobileNet.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_MobileNet.add(Dropout(0.2))
model_MobileNet.add(Dense(4, activation='softmax'))
# Model summary
print("Model Summary (MobileNetV2):")
model_MobileNet.summary()
print()
# Compile the model
model_MobileNet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history_MobileNet = model_MobileNet.fit(train_data, epochs=EPOCHS, validation_data=test_generator, class_weight=class_weights)
# Validate the model
val_loss_MobileNet, val_accuracy_MobileNet = model_MobileNet.evaluate(test_generator, steps=len(test_generator))
print(f'Validation Loss: {val_loss_MobileNet:.4f}')
print(f'Validation Accuracy: {val_accuracy_MobileNet:.4f}')

##**Transfer Learning using DenseNet121**

- DenseNet, introduced in **2017**, is a convolutional neural network (CNN) architecture known for its efficient use of parameters and its ability to achieve high accuracy with relatively fewer layers.

- Unlike traditional CNNs, where each layer's output is passed to the next layer, DenseNet connects every layer to every other layer after it. This dense
connectivity pattern enhances information flow and gradient propagation, leading to improved performance.

![Image](https://raw.githubusercontent.com/MazenMTULab/ML_COURSE_RESOURCES/refs/heads/main/Figs/DenseNet.webp)

In [ ]:
%%time
from tensorflow.keras.applications import DenseNet121
# Load the pre-trained DenseNet121 model without the top classification layer
base_model_DenseNet = DenseNet121(weights='imagenet', include_top=False, input_shape=(image_height, image_width, 3))
# Set the layers of the base model as non-trainable (freeze them)
for layer in base_model_DenseNet.layers:
    layer.trainable = False
# Create a new model and add the DenseNet121 base model
model_DenseNet = Sequential()
model_DenseNet.add(base_model_DenseNet)
# Add a global average pooling layer and output layer for classification
model_DenseNet.add(GlobalAveragePooling2D())
model_DenseNet.add(Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_DenseNet.add(Dropout(0.4))
model_DenseNet.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))

model_DenseNet.add(Dropout(0.2))
model_DenseNet.add(Dense(4, activation='softmax'))
# Model summary
print("Model Summary (DenseNet121):")
model_DenseNet.summary()
print()
# Compile the model
model_DenseNet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Train the model
history_DenseNet = model_DenseNet.fit(train_data, epochs=EPOCHS, validation_data=test_generator, callbacks=[early_stopping], class_weight=class_weights)
# Validate the model
val_loss_DenseNet, val_accuracy_DenseNet = model_DenseNet.evaluate(test_generator, steps=len(test_generator))
print(f'Validation Loss: {val_loss_DenseNet:.4f}')
print(f'Validation Accuracy: {val_accuracy_DenseNet:.4f}')

##**Transfer Learning using InceptionV3**

- InceptionV3 is a convolutional neural network (CNN) architecture introduced in **2015** that is known for its depth, width, and computational efficiency. It builds upon the ideas of the Inception modules introduced in earlier Inception versions, incorporating several enhancements to improve performance.

- **Inception Modules:** The core building block of InceptionV3 is the Inception module. It consists of a parallel combination of different convolutional filters with different sizes (1x1, 3x3, 5x5) and a pooling layer. This allows the network to capture features at different scales.

![Image](https://raw.githubusercontent.com/MazenMTULab/ML_COURSE_RESOURCES/refs/heads/main/Figs/inception-modules.webp)

In [ ]:
%%time
from tensorflow.keras.applications import InceptionV3
# Load the pre-trained InceptionV3 model without the top classification layer
base_model_Inception = InceptionV3(weights='imagenet', include_top=False, input_shape=(image_height, image_width, 3))
# Set the layers of the base model as non-trainable (freeze them)
for layer in base_model_Inception.layers:
    layer.trainable = False
# Create a new model and add the InceptionV3 base model
model_Inception = Sequential()
model_Inception.add(base_model_Inception)
# Add a global average pooling layer and output layer for classification
model_Inception.add(GlobalAveragePooling2D())

model_Inception.add(Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_Inception.add(Dropout(0.4))
model_Inception.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_Inception.add(Dropout(0.2))
model_Inception.add(Dense(4, activation='softmax'))
# Model summary
print("Model Summary (InceptionV3):")
model_Inception.summary()
print()
# Compile the model
model_Inception.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Train the model with EarlyStopping
history_Inception = model_Inception.fit(train_data, epochs=EPOCHS, validation_data=test_generator, callbacks=[early_stopping], class_weight=class_weights)
# Validate the model
val_loss_Inception, val_accuracy_Inception = model_Inception.evaluate(test_generator, steps=len(test_generator))
print(f'Validation Loss: {val_loss_Inception:.4f}')
print(f'Validation Accuracy: {val_accuracy_Inception:.4f}')

##**Transfer Learning using ResNet**

- ResNet (Residual Network), introduced in **2015**, is a type of convolutional neural network (CNN) architecture that has significantly impacted the field of deep learning.

- The key innovation in ResNet is the introduction of residual blocks, which allow the network to learn residual functions instead of the entire underlying mapping. This enables the training of extremely deep networks without suffering from the vanishing gradient problem.

**Traditional CNN** suffers from **vanishing gradient** problem where gradients become too small, preventing weight updates.

**A standard residual block in ResNet has:**

- Two convolutional layers with Batch Normalization and ReLU activation.

- **A shortcut (skip connection)** that bypasses the convolutional layers and directly adds the input to the output. This means the network learns the difference (residual) between the input and the output, making it easier to optimize.



In [ ]:
%%time
from tensorflow.keras.applications import ResNet50

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import regularizers

# Load the pre-trained ResNet50 model without the top classification layer
base_model_ResNet = ResNet50(weights='imagenet', include_top=False, input_shape=(image_height, image_width, 3))
# Set the layers of the base model as non-trainable (freeze them)
for layer in base_model_ResNet.layers:
    layer.trainable = False
# Create a new model and add the ResNet50 base model
model_ResNet = Sequential()
model_ResNet.add(base_model_ResNet)
# Add a global average pooling layer and output layer for classification
model_ResNet.add(GlobalAveragePooling2D())
model_ResNet.add(Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_ResNet.add(Dropout(0.4))
model_ResNet.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_ResNet.add(Dropout(0.2))
model_ResNet.add(Dense(4, activation='softmax'))
# Model summary
print("Model Summary (ResNet50):")
model_ResNet.summary()
print()
# Compile the model
model_ResNet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Train the model with EarlyStopping
history_ResNet = model_ResNet.fit(train_data, epochs=EPOCHS, validation_data=test_generator, callbacks=[early_stopping], class_weight=class_weights)

# Validate the model
val_loss_ResNet, val_accuracy_ResNet = model_ResNet.evaluate(test_generator, steps=len(test_generator))
print(f'Validation Loss: {val_loss_ResNet:.4f}')
print(f'Validation Accuracy: {val_accuracy_ResNet:.4f}')

##**Transfer Learning using EfficientNetB0**

EfficientNet is a family of convolutional neural network (CNN) architectures designed to achieve state-of-the-art accuracy with significantly fewer computational resources compared to previous models.

It was developed by researchers at **Google Brain** in 2019.

It introduces a novel scaling method that uniformly scales the network's depth, width, and resolution.

In [ ]:
%%time
from tensorflow.keras.applications import EfficientNetB0

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import regularizers

# Load the pre-trained EfficientNetB0 model without the top classification layer
base_model_EfficientNet = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(image_height, image_width, 3))
# Set the layers of the base model as non-trainable (freeze them)
for layer in base_model_EfficientNet.layers:
    layer.trainable = False
# Create a new model and add the EfficientNetB0 base model
model_EfficientNet = Sequential()
model_EfficientNet.add(base_model_EfficientNet)
# Add a global average pooling layer and output layer for classification
model_EfficientNet.add(GlobalAveragePooling2D())
model_EfficientNet.add(Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_EfficientNet.add(Dropout(0.4))
model_EfficientNet.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_EfficientNet.add(Dropout(0.2))
model_EfficientNet.add(Dense(4, activation='softmax'))
# Model summary

model_NASNet.add(GlobalAveragePooling2D())

model_NASNet.add(Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_NASNet.add(Dropout(0.4))
model_NASNet.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_NASNet.add(Dropout(0.2))
model_NASNet.add(Dense(4, activation='softmax'))

print("Model Summary (EfficientNetB0):")
model_EfficientNet.summary()
print()
# Compile the model
model_EfficientNet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Train the model with EarlyStopping
history_EfficientNet = model_EfficientNet.fit(train_data, epochs=EPOCHS, validation_data=test_generator, callbacks=[early_stopping],class_weight=class_weights)
# Validate the model
val_loss_EfficientNet, val_accuracy_EfficientNet = model_EfficientNet.evaluate(test_generator, steps=len(test_generator))
print(f'Validation Loss: {val_loss_EfficientNet:.4f}')
print(f'Validation Accuracy: {val_accuracy_EfficientNet:.4f}')

##**Transfer Learning using NASNetMobile**

NASNet is a convolutional neural network (CNN) architecture that was designed using neural architecture search (NAS) techniques. This means that the network's
architecture was not manually designed by humans but rather was automatically
generated by a machine learning algorithm.

NASNet was developed by **Google Brain** in **2017**.

In [ ]:
%%time

from tensorflow.keras.applications import NASNetMobile

import tensorflow as tf
from tensorflow.keras.applications import NASNetMobile
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import regularizers

# Load the pre-trained NASNetMobile model without top classification layer
base_model_NASNet = NASNetMobile(weights='imagenet', include_top=False, input_shape=(image_height, image_width, 3))
# Set the layers of the base model as non-trainable (freeze them)
for layer in base_model_NASNet.layers:
    layer.trainable = False
# Create a new model and add the NASNetMobile base model
model_NASNet = Sequential()
model_NASNet.add(base_model_NASNet)
# Add a global average pooling layer and output layer for classification
model_NASNet.add(GlobalAveragePooling2D())

model_NASNet.add(Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_NASNet.add(Dropout(0.4))
model_NASNet.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model_NASNet.add(Dropout(0.2))
model_NASNet.add(Dense(4, activation='softmax'))
# Model summary
print("Model Summary (NASNetMobile):")
model_NASNet.summary()
print()
# Compile the model
model_NASNet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_NASNet = model_NASNet.fit(train_data, epochs=EPOCHS, validation_data=test_generator, callbacks=[early_stopping], class_weight=class_weights)
#Validate the model
val_loss_NASNet, val_accuracy_NASNet = model_NASNet.evaluate(test_generator, steps=len(test_generator))
print(f'Validation Loss: {val_loss_NASNet:.4f}')
print(f'Validation Accuracy: {val_accuracy_NASNet:.4f}')

##**Model Performance Comparison**

In [ ]:
data = {
'VGG16': val_accuracy_VGG16,
'MobileNet': val_accuracy_MobileNet,
'DenseNet': val_accuracy_DenseNet,
'Inception': val_accuracy_Inception,
'ResNetNASNet' : val_accuracy_ResNet,
'EfficientNet' : val_accuracy_EfficientNet,
'NASNet' : val_accuracy_NASNet
}

df = pd.DataFrame.from_dict(data, orient='index', columns=['Accuracy'])
df = df.reset_index().rename(columns={'index': 'Model'})

plt.figure(figsize=[15, 5])
# Create bar chart
sns.barplot(x='Model', y='Accuracy', data=df)
# Add labels to bars
ax = plt.gca()
for bar in ax.containers:
    ax.bar_label(bar, label_type='edge', labels=[f"{x:.1%}" for x in bar.
    datavalues], fontsize=20)
    # Adjust the layout
    plt.tight_layout()
    plt.show()

##**Prediction Result Samples**

### MobileNet

In [ ]:
test_generator.reset()
img, label = next(test_generator)
prediction = model_MobileNet.predict(img)
test_pred_classes = np.argmax(prediction, axis=1)
plt.figure(figsize=[20, 20])
for i in range(20):
    plt.subplot(5, 4, i+1)
    plt.imshow(img[i])
    plt.axis('off')
    plt.title("Label : {}\n Prediction : {} {:.1f}%".format(class_names[np.argmax(label[i])], class_names[test_pred_classes[i]], 100 * np.max(prediction[i])))
plt.show()

### Inception V3

In [ ]:
test_generator.reset()
img, label = next(test_generator)
prediction = model_Inception.predict(img)
test_pred_classes = np.argmax(prediction, axis=1)
plt.figure(figsize=[20, 20])
for i in range(20):
    plt.subplot(5, 4, i+1)
    plt.imshow(img[i])
    plt.axis('off')
    plt.title("Label : {}\n Prediction : {} {:.1f}%".format(class_names[np.argmax(label[i])], class_names[test_pred_classes[i]], 100 * np.max(prediction[i])))
plt.show()

